# Featureselectie en boosting

Lesbasis: **4 — Models**, dia 42 en 49 voor geoptimaliseerde boosting en regularisatie. Alleen 4.100 trainingsrijen; dit is verdere ontwikkeling op eerder gebruikte data.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'results_v3/run.json').exists())
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, scores
from improve_v3 import experiments, InputFeatures, WEIGHTS
from build_v3_reports import label
X, y, ids, folds = development_data()
summary = pd.read_csv(HERE/'results_v3/summary.csv').set_index('model')
details = pd.read_csv(HERE/'results_v3/outer_fold_metrics.csv')
predictions = pd.read_csv(HERE/'results_v3/outer_oof_predictions.csv', float_precision='round_trip')
selections = json.loads((HERE/'results_v3/selections.json').read_text())
assert len(summary) == 14 and summary.folds.eq(5).all()


## Eerst de datavoorbereiding

Een veld intern weglaten verandert het publieke formulier met elf velden niet. De keuze gebeurt binnen training. Het label komt nooit in de invoer terecht. Voor LightGBM leren we de categorievocabulaire uitsluitend van training; CatBoost verwerkt categorische tekstwaarden.

In [2]:
display(X.isna().mean().sort_values(ascending=False).rename('fractie_ontbrekend').to_frame())
for family,(pipeline,grid) in experiments().items():
    print(family, pipeline)
    print('zoekruimte:', grid)

rf_selection Pipeline(steps=[('inputs', InputFeatures()),
                ('preprocessor',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001DFB110A660>),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                 

,fractie_ontbrekend
spore_print_color,0.948293
stem_surface,0.673659
cap_diameter,0.398780
stem_height,0.206341
habitat,0.202683
season,0.195122
gill_color,0.194634
ring_type,0.113415
cap_shape,0.080732
jumbled_noise_1,0.080488


## De trainingsstappen zoals in de oefeningen

Onderstaande optionele cel herhaalt één RF-buitenfold en schrijft geen bestaand resultaat over. De volledige echte run staat in `improve_v3.py`; alle vijf folds en alle families zijn al uitgevoerd voor de tabellen hier. Zet de schakelaar alleen aan als je deze ene fit opnieuw wilt doen.

In [3]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold

RERUN_ONE_FOLD = False
if RERUN_ONE_FOLD:
    train, validation = folds[0]
    pipeline, grid = experiments()['rf_selection']
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=9411)
    search = GridSearchCV(clone(pipeline), grid, scoring='f1_macro', cv=inner_cv, n_jobs=1, error_score='raise')
    search.fit(X.iloc[train], y.iloc[train])
    probability = search.best_estimator_.predict_proba(X.iloc[validation])[:,1]
    raw_label = search.best_estimator_.predict(X.iloc[validation])
    display(search.best_params_)
    display(scores(y.iloc[validation], probability, prediction=raw_label))
else:
    print('Rapportmodus: hieronder staan de bewaarde resultaten van de complete training.')

Rapportmodus: hieronder staan de bewaarde resultaten van de complete training.


## Bekijk de werkelijke selectie per buitenfold

Deze tabel toont gekozen hyperparameters en binnen-CV-scores, geen onafhankelijke eindprestaties. De buitenfold wordt pas na de selectie voorspeld.

In [4]:
display(pd.DataFrame([s for s in selections if s['family']!='voting'])[['family','fold','parameters','inner_cv_macro_f1','f1_threshold','fpr_threshold']])

,family,fold,parameters,inner_cv_macro_f1,f1_threshold,fpr_threshold
0,rf_selection,1,"{'classifier__min_samples_leaf': 1, 'inputs__d...",0.761595,0.49,0.462181
1,lightgbm,1,"{'classifier__min_child_samples': 20, 'classif...",0.743924,0.49,0.561955
2,catboost,1,{'classifier__cat_features': ['spore_print_col...,0.729771,0.51,0.539973
3,rf_selection,2,"{'classifier__min_samples_leaf': 1, 'inputs__d...",0.759517,0.48,0.464486
4,lightgbm,2,"{'classifier__min_child_samples': 20, 'classif...",0.747026,0.53,0.551434
5,catboost,2,{'classifier__cat_features': ['spore_print_col...,0.740081,0.50,0.523203
6,rf_selection,3,"{'classifier__min_samples_leaf': 1, 'inputs__d...",0.759108,0.50,0.462261
7,lightgbm,3,"{'classifier__min_child_samples': 60, 'classif...",0.750096,0.52,0.555387
8,catboost,3,{'classifier__cat_features': ['spore_print_col...,0.748857,0.49,0.514227
9,rf_selection,4,"{'classifier__min_samples_leaf': 1, 'inputs__d...",0.745850,0.44,0.473325


## Vergelijk de uiteindelijke procedures

De standaardgrens 0,5 en de binnen training afgestemde drempels staan afzonderlijk. Gebruik de cijfers van de procedure die je werkelijk toepast.

In [5]:
view = summary[['macro_f1_mean','recall_p_mean','fn_total','fp_total','accuracy_mean']].copy()
view.index = [label(n) for n in view.index]
display(view.round(4))

,macro_f1_mean,recall_p_mean,fn_total,fp_total,accuracy_mean
Oude RF300,0.7619,0.6044,614,256,0.7878
Kandidaat v2,0.7691,0.6682,515,356,0.7876
RF + featureselectie + standaardgrens,0.7749,0.6289,576,251,0.7983
RF + featureselectie + F1-drempel,0.7690,0.6566,533,332,0.7890
RF + featureselectie + FPR-budget,0.7745,0.6611,526,319,0.7939
LightGBM + standaardgrens,0.7611,0.6875,485,430,0.7768
LightGBM + F1-drempel,0.7629,0.6714,510,390,0.7805
LightGBM + FPR-budget,0.7658,0.6489,545,332,0.7861
CatBoost + standaardgrens,0.7556,0.6675,516,414,0.7732
CatBoost + F1-drempel,0.7544,0.6488,545,380,0.7744


Voor opnieuw trainen: voer `reproduce_v3.py` uit met de vastgelegde dependencies. De rapportcellen lezen de complete echte zoekuitvoer; ze herhalen niet stilzwijgend honderden fits.